In [ ]:
from torch import nn, permute
import torch
import torch.nn.functional as F
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

IMG_H=np.short(512)
IMG_W=np.short(512)
IMG_C=1
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
from pytorch_nndct.apis import Inspector

In [ ]:
from CNetPlusScalar import CNetPlusScalar
model = CNetPlusScalar()

In [ ]:
# Specify a target name or fingerprint you want to deploy on
target = "DPUCZDX8G_ISA1_B4096"
# Initialize inspector with target
inspector = Inspector(target)

In [ ]:
# Start to inspect the float model
# Note: visualization of inspection results relies on the dot engine.If you don't install dot successfully, set 'image_format = None' when inspecting.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load('pre_trained_w.pt'))

In [ ]:
dummy_input = (torch.randn(1, IMG_C, IMG_H, IMG_W), torch.randn(1, 1))
inspector.inspect(model, dummy_input, device=device, output_dir=f"{output_dir}/inspect", image_format="png") 

In [ ]:
# Show the dot image
from IPython.display import Image
Image(f"{output_dir}/inspect/inspect_{target}.png")

In [ ]:
from pytorch_nndct.apis import torch_quantizer
import pathlib
import typing

def quantize_model(model: nn.Module, quant_mode: str, device: torch.device, quant_stub: typing.List[torch.Tensor],
                   quant_dataset: Dataset, output_path: pathlib.Path, batch_size: int,
                   num_workers: int = 4):
    """
    Configures quantization functions and invokes evaluation to generate quantized model.
    Args:
        model (torch.nn.Module):  Float model to quantize.
        quant_mode (str): Quantizer mode.  Either 'calib' or 'test'.
        device (torch.device): Device to use for model quantization.
        quant_stub (torch.Tensor): Random tensor with same dimensionality as real input.
        quant_dataset (torch.utils.data.Dataset):  Dataset of samples used for quantization.
        output_path (pathlib.Path): Output path for quantization artifacts.
        batch_size (int): Batch size for quantization.
        num_workers (int): Number of workers for the DataLoader.
    Returns:
        None for calibration mode, quantized inference model for test mode.
    """
    if quant_mode == 'calib':
        actual_batch_size = batch_size
    elif quant_mode == 'test':
        actual_batch_size = 1
    else:
        raise ValueError('ERROR: Unknown quant_mode {}!'.format(quant_mode))
    quant_loader = DataLoader(quant_dataset, batch_size=actual_batch_size, shuffle=True, num_workers=num_workers,
                                  pin_memory=True)
    quantizer = torch_quantizer(quant_mode, model, quant_stub, output_dir=str(output_path), device=device)
    quant_model = quantizer.quant_model
    evaluate_model(quant_model, quant_loader, device, quantizer, quant_mode, output_path)
    if quant_mode == 'calib':
        return None
    elif quant_mode == 'test':
        return quant_model

def evaluate_model(model: nn.Module, dataloader: DataLoader, device: torch.device, quantizer: torch_quantizer,
                   quant_mode: str, output_path: pathlib.Path):
    """
    Evaluates a quantizable model on the specified device according to the supplied training split subset.
    Required to calibrate the quantization output artifacts.
    Args:
        model (tnn.Module): Model to quantize.
        dataloader (torch.utils.data.DataLoader): Dataloader with quantization subset of training split.
        device (torch.device): Compute device to quantize on.
        quantizer (pytorch_nndct.apis.torch_quantizer): Xilinx quantization module.
        quant_mode (str): Quantization mode to run in.  Either 'calib' or 'test'.
        output_path (pathlib.Path): Output path for quantization artifacts.
    """
    model.eval()
    with torch.no_grad():
        for image, scalar, _ in dataloader:
            out_tensor = model(image, scalar)
        if quant_mode == 'test':
            print('Network output shape = {}'.format(out_tensor[0].shape))

    if quant_mode == 'calib':
        quantizer.export_quant_config()
    if quant_mode == 'test':
        quantizer.export_xmodel(deploy_check=True, output_dir=str(output_path))

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, num_samples: int):
        self.img = torch.randn(num_samples, 1, 512, 512)
        self.scalar = torch.randn(num_samples, 1)
        self.label = torch.randn(num_samples, 1)

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        image = self.img[idx]
        scalar = self.scalar[idx]
        label = self.label[idx]
        return image, scalar, label

dataset = CustomDataset(10)

In [ ]:
# Quantization
quantize_model(model, 'calib', device, dummy_input, dataset, output_dir, 1)
quantize_model(model, 'test', device, dummy_input, dataset, output_dir, 1)

In [ ]:
old_weights = torch.load('pre_trained_w.pt')
new_weights = model.state_dict()
for key in old_weights.keys():
    if key in new_weights:
        if(key.endswith(".weight") or key.endswith(".bias")):
            if torch.all(old_weights[key] == new_weights[key]):
                print(f"{key}: Weights are exactly equal")
            else:
                mse = torch.mean((old_weights[key] - new_weights[key]) ** 2).item()
                print(f"MSE for {key}: {mse}")
        else:
            print(f"Key {key} is not a weight or bias")
    else:
        print(f"Key {key} not found in new_weights")

In [ ]:
!vai_c_xir --xmodel output/CNetPlusScalar_int.xmodel --arch /opt/vitis_ai/compiler/arch/DPUCZDX8G/ZCU104/arch.json --net_name zcu104_CNetPlusScalar